In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Global Configuration
DEVICE_CONFIG = "cpu"  # "auto", "cpu", or "mps"
TORCH_DTYPE = torch.float32  # "auto" or specific type


def get_model_and_tokenizer():
    """Load model/tokenizer with proper settings"""
    # Use simplified model directory name to avoid path conflicts
    model_dir = "./Qwen2.5-0.5B-Instruct"  # Simplified path

    if not os.path.exists(model_dir):
        print("Downloading and saving model locally...")
        os.makedirs(model_dir, exist_ok=True)

        # Load with trust_remote_code for custom components
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=TORCH_DTYPE,
            device_map=DEVICE_CONFIG,
            trust_remote_code=True
        )
        model.save_pretrained(model_dir)

        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        tokenizer.save_pretrained(model_dir)
    else:
        print("Loading local model...")
        model = AutoModelForCausalLM.from_pretrained(
            model_dir,
            torch_dtype=TORCH_DTYPE if TORCH_DTYPE != "auto" else None,
            device_map=DEVICE_CONFIG,
            trust_remote_code=True
        )

        if DEVICE_CONFIG == "mps":
            model = model.to(torch.device("mps"))

        # Load tokenizer with trust_remote_code
        tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            trust_remote_code=True
        )

    return model, tokenizer


# Initialize once and reuse
model, tokenizer = get_model_and_tokenizer()

Loading local model...


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import requests
import json
import re


def fetch_webpage_content(url):
    """Fetch webpage content with error handling"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        print(f"Fetched URL: {url}")
        print(f"response.text: {response.text}")
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching URL: {e}")
        return None


def generate_response(messages):
    """Generate response using the model with chat template"""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        temperature=0.7
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


def parse_json(json_str):
    """Parse and validate JSON output with error recovery"""
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # Try to extract JSON from markdown code blocks
        json_match = re.search(r'```json\n(.*?)\n```', json_str, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(1))
            except json.JSONDecodeError:
                pass
        # Try to find the first valid JSON structure
        json_match = re.search(r'(\{.*\}|\[.*\])', json_str, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(0))
            except json.JSONDecodeError:
                pass
        return None


def news_pipeline(url, prompt):
    """Main pipeline to process news from URL to structured JSON"""
    # Step 1: Fetch webpage content
    content = fetch_webpage_content(url)
    if not content:
        return {"error": "Failed to fetch webpage content"}

    # Step 2: Extract news list
    extraction_messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        # Truncate to avoid token limits
        {"role": "user",
            "content": f"{prompt}\n\nWebsite Content:\n{content[:15000]}"}
    ]
    news_list = generate_response(extraction_messages)

    print(f"news_list: {news_list}")

    # Step 3: Convert to JSON
    json_prompt = """Convert this news list into a JSON array of objects with:
    - title (string)
    - summary (string)
    - date (string in YYYY-MM-DD format)
    - url (string if available)
    
    News List:
    """
    conversion_messages = [
        {"role": "system", "content": "You are a JSON formatting assistant. Return only valid JSON."},
        {"role": "user", "content": f"{json_prompt}\n{news_list}"}
    ]
    json_output = generate_response(conversion_messages)

    # Step 4: Parse and validate JSON
    parsed = parse_json(json_output)
    return parsed if parsed else {"error": "Invalid JSON format", "raw_output": json_output}


# Example usage
if __name__ == "__main__":
    result = news_pipeline(
        url="https://www.bbc.com/news",
        prompt="List the top 5 current news headlines with their summaries"
    )
    print(json.dumps(result, indent=2))

Fetched URL: https://www.bbc.com/news
response.text: <!DOCTYPE html><html lang="en-GB"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><title>Home - BBC News</title><meta property="og:title" content="Home - BBC News"/><meta name="twitter:title" content="Home - BBC News"/><meta name="description" content="Visit BBC News for up-to-the-minute news, breaking news, video, audio and feature stories. BBC News provides trusted World and UK news as well as local and regional perspectives. Also entertainment, business, science, technology and health news."/><meta property="og:description" content="Visit BBC News for up-to-the-minute news, breaking news, video, audio and feature stories. BBC News provides trusted World and UK news as well as local and regional perspectives. Also entertainment, business, science, technology and health news."/><meta name="twitter:description" content="Visit BBC News for up-to-the-minute news, breaking news, video, audio and feature 